In [9]:
from PIL import Image
import os
from tqdm import tqdm
import numpy as np
import shutil
from sklearn.model_selection import train_test_split

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [10]:
# source = '/media/data2/dataset/GAN_ImageData/StyleGAN2_256/1'
# real_source = '/media/data2/dataset/GAN_ImageData/StyleGAN_256/'
# real_category = ['train', 'validation', 'test']
# target = '/media/data2/dataset/GAN_ImageData/StyleGAN2_256/'

source = '/home/vincent/Downloads/FAKE/StableDiffusion_256'
real_source = '/home/vincent/Downloads/REAL/FFHQ_256/'
real_category = ['train', 'validation', 'test']
target = '/home/vincent/Downloads/FAKE/'

In [11]:
fake_set = os.listdir(source)

In [ ]:
length_list = []
for c in real_category:
    c += '/0'
    source_dir = os.path.join(real_source, c)
    target_dir = os.path.join(target, c)
    images = os.listdir(source_dir)
    length_list.append(len(images))
    
    for i in tqdm(images):
        img_dir = os.path.join(source_dir, i)
        shutil.copy2(img_dir, target_dir)

In [ ]:
fake_train, fake_test = train_test_split(fake_set, test_size=30000)

In [ ]:
fake_train, fake_val = train_test_split(fake_train, test_size=0.2)

In [12]:
from sklearn.model_selection import train_test_split

# Split full dataset into Train/Val pool (80%) and Test set (20%)
# Use a common variable name for the temporary pool, e.g., 'temp_train'
temp_train, fake_test = train_test_split(fake_set, test_size=0.2, random_state=42)

In [13]:
# Split the Train/Val pool: 12.5% of the pool is 10% of the original data.
fake_train, fake_val = train_test_split(temp_train, test_size=0.125, random_state=42)

In [14]:
print(len(fake_train), len(fake_val), len(fake_test))
fake_list = [fake_train, fake_val, fake_test]

23100 3300 6600


In [15]:
for idx in range(3):
    tmp = fake_list[idx]
    category = real_category[idx]
    for i in tqdm(tmp):
        c_dir = os.path.join(target, category)
        c_dir = os.path.join(c_dir, '1')
        os.makedirs(c_dir, exist_ok=True)  # Ensure the directory exists
        img_dir = os.path.join(c_dir, i)
        source_img = os.path.join(source, i)
        shutil.copy2(source_img, img_dir)

100%|██████████| 6600/6600 [01:57<00:00, 56.08it/s]


In [18]:
# 1. Get list of all real images from the source directory
real_set = os.listdir(real_source)

# 2. Split the list using fixed sizes for test and validation sets
# real_train, real_test = train_test_split(real_set, test_size=30000)
# real_train, real_val = train_test_split(real_train, test_size=3900)

# Alternatively, use proportions to split
temp_train_val, real_test = train_test_split(real_set, test_size=0.2, random_state=42)
real_train, real_val = train_test_split(temp_train_val, test_size=0.125, random_state=42)


real_list = [real_train, real_val, real_test]

# 3. Copy the split real images to the target directories
for idx in range(3):
    tmp = real_list[idx]
    category = real_category[idx] # e.g., 'train', 'validation', 'test'
    
    # Target directory structure for real images is .../category/0
    c_dir = os.path.join(target, category)
    c_dir = os.path.join(c_dir, '0')
    os.makedirs(c_dir, exist_ok=True)
    
    for i in tqdm(tmp):
        source_img = os.path.join(real_source, i)
        img_dir = os.path.join(c_dir, i)
        shutil.copy2(source_img, img_dir)

100%|██████████| 14000/14000 [02:30<00:00, 93.06it/s] 


In [17]:
# 70,000 images total
real_set = os.listdir(real_source)
# Step 1: Split into a temporary pool (80%) and a final Test set (20%)
# Use test_size=0.2 (20%) for the final test set
temp_train_val, real_test = train_test_split(real_set, test_size=0.2, random_state=42)
# real_test will have 14,000 images (20% of 70,000)
# temp_train_val will have 56,000 images (80% of 70,000)


# Step 2: Split the temporary pool (80%) into final Train (70%) and Val (10%)
# The Validation set (10% of the total) must be 1/8th of the remaining 80% pool.
# 10% / 80% = 0.125
real_train, real_val = train_test_split(temp_train_val, test_size=0.125, random_state=42)
# real_val will have 7,000 images (12.5% of 56,000 = 10% of 70,000)
# real_train will have 49,000 images (87.5% of 56,000 = 70% of 70,000)
# **This gives a 70% / 10% / 20% split, matching your fake data ratios.**

# Final Split Check:
# real_train: 49,000
# real_val: 7,000
# real_test: 14,000
# Total: 70,000